<div style="background:#512888;border-left:10px solid #e3b747;border-radius:13px;padding:31px 32px 29px;color:white;">
  <div style="font-size:12px;letter-spacing:2.1px;font-weight:750;color:#f8dd9b;">STEPHEN F. AUSTIN STATE UNIVERSITY &nbsp; / &nbsp; CSCI 1462</div>
  <div style="font-size:38px;font-weight:790;line-height:1.16;margin-top:15px;color:white;">Combining Tables in pandas</div>
  <div style="font-size:18px;color:#f0e7ff;margin-top:12px;">Concatenation &bull; Merging &bull; Join Types &bull; Common Pitfalls</div>
  <div style="display:inline-block;background:#ffffff19;border:1px solid #ffffff55;border-radius:24px;margin-top:21px;padding:7px 14px;font-size:13px;color:#fff;">INTRODUCTION TO DATA ANALYTICS &nbsp;&middot;&nbsp; LECTURE TUTORIAL</div>
</div>

<div style="background:#f5f1fa;border-left:5px solid #512888;border-radius:7px;padding:17px 20px;">
  <div style="font-size:19px;color:#512888;font-weight:750;">Learning outcomes</div>
  <div style="margin-top:7px;line-height:1.72;color:#2f2935;">By the end of this lecture, students will be able to combine DataFrames by rows or columns, distinguish index alignment from key-based merging, compare all five join types, use multi-column keys, and diagnose duplicated or missing records in merge results.</div>
</div>

**Using this notebook:** Read the explanation and run the Python examples from top to bottom using **Shift + Enter**. The tables are also saved as outputs for easy review. All examples use small, self-contained datasets; no external CSV files are needed.

<div style="display:flex;gap:12px;flex-wrap:wrap;margin:13px 0 22px 0;">
 <div style="flex:1;min-width:155px;background:#fff;border:1px solid #ddd3eb;border-top:4px solid #e3b747;border-radius:8px;padding:14px;"><div style="font-size:12px;color:#625676;font-weight:750;letter-spacing:1px;">PART 01</div><div style="color:#512888;font-size:17px;font-weight:720;margin-top:4px;">Concatenation</div><div style="font-size:13px;color:#4c4553;margin-top:4px;">Stack rows and align columns</div></div>
 <div style="flex:1;min-width:155px;background:#fff;border:1px solid #ddd3eb;border-top:4px solid #e3b747;border-radius:8px;padding:14px;"><div style="font-size:12px;color:#625676;font-weight:750;letter-spacing:1px;">PART 02</div><div style="color:#512888;font-size:17px;font-weight:720;margin-top:4px;">Table Joins</div><div style="font-size:13px;color:#4c4553;margin-top:4px;">Inner, left, right, outer, cross</div></div>
 <div style="flex:1;min-width:155px;background:#fff;border:1px solid #ddd3eb;border-top:4px solid #e3b747;border-radius:8px;padding:14px;"><div style="font-size:12px;color:#625676;font-weight:750;letter-spacing:1px;">PART 03</div><div style="color:#512888;font-size:17px;font-weight:720;margin-top:4px;">Merge Safety</div><div style="font-size:13px;color:#4c4553;margin-top:4px;">Duplicates, nulls, key mismatches</div></div>
 <div style="flex:1;min-width:155px;background:#fff;border:1px solid #ddd3eb;border-top:4px solid #e3b747;border-radius:8px;padding:14px;"><div style="font-size:12px;color:#625676;font-weight:750;letter-spacing:1px;">PART 04</div><div style="color:#512888;font-size:17px;font-weight:720;margin-top:4px;">Worked Example</div><div style="font-size:13px;color:#4c4553;margin-top:4px;">Campus meal-swipes tables</div></div>
</div>

In [2]:
import pandas as pd

print("pandas version:", pd.__version__)

pandas version: 2.2.3


<div style="border-left:6px solid #512888;padding:13px 18px;background:#f4f0fa;border-radius:5px;margin:24px 0 14px 0"><div style="color:#512888;font-size:22px;font-weight:750">01 · Two ways to combine tables</div><div style="color:#4b4655;font-size:14px;margin-top:5px">Concatenate by position/index; merge by matching key values</div></div>

| Operation | Primary purpose | Alignment rule |
|---|---|---|
| `pd.concat([a, b], axis=0)` | Stack rows | Columns are aligned by column labels |
| `pd.concat([a, b], axis=1)` | Place columns side by side | Rows are aligned by **index labels** |
| `pd.merge(a, b, on="key")` | Join related records | Rows are matched using **key values** |

**Mental model:** `concat()` puts pieces alongside one another; `merge()` decides which records belong together.

<div style="border-left:6px solid #512888;padding:13px 18px;background:#f4f0fa;border-radius:5px;margin:24px 0 14px 0"><div style="color:#512888;font-size:22px;font-weight:750">02 · Vertical concatenation</div><div style="color:#4b4655;font-size:14px;margin-top:5px">Stack one DataFrame below another</div></div>

In [3]:
fall = pd.DataFrame({"Name": ["Kevin", "Tyler", "Kyla"], "Age": [22, 35, 21]})
spring = pd.DataFrame({"Name": ["Justin", "Jacob"], "Age": [20, 19]})
print("FALL"); display(fall)
print("SPRING"); display(spring)

FALL


,Name,Age
0,Kevin,22
1,Tyler,35
2,Kyla,21


SPRING


,Name,Age
0,Justin,20
1,Jacob,19


In [4]:
students = pd.concat([fall, spring], ignore_index=True)
display(students)

,Name,Age
0,Kevin,22
1,Tyler,35
2,Kyla,21
3,Justin,20
4,Jacob,19


`ignore_index=True` creates new row labels `0, 1, 2, ...`. Without it, the original indexes are retained, potentially producing repeated labels.

In [5]:
print("Without ignore_index:")
display(pd.concat([fall, spring]))

Without ignore_index:


,Name,Age
0,Kevin,22
1,Tyler,35
2,Kyla,21
0,Justin,20
1,Jacob,19


<div style="background:#fffaf0;border-left:4px solid #D9B45B;padding:11px 16px;border-radius:4px;margin:10px 0"><b style="color:#614616">When column names differ</b><br>By default, vertical concatenation includes the <b>union of column names</b> and fills missing entries with <code>NaN</code>. Set <code>join="inner"</code> to retain only column names shared by all inputs.</div>

<div style="border-left:6px solid #512888;padding:13px 18px;background:#f4f0fa;border-radius:5px;margin:24px 0 14px 0"><div style="color:#512888;font-size:22px;font-weight:750">03 · Horizontal concatenation</div><div style="color:#4b4655;font-size:14px;margin-top:5px">Combine columns using the row index</div></div>

In [6]:
basic = pd.DataFrame({"Name": ["Kevin", "Tyler", "Kyla"]}, index=[101, 102, 103])
major = pd.DataFrame({"Major": ["Computer Science", "Mathematics", "Biology"]}, index=[103, 101, 104])
print("BASIC"); display(basic)
print("MAJOR"); display(major)

BASIC


,Name
101,Kevin
102,Tyler
103,Kyla


MAJOR


,Major
103,Computer Science
101,Mathematics
104,Biology


In [7]:
print("Outer alignment: all index labels")
display(pd.concat([basic, major], axis="columns"))
print("Inner alignment: shared index labels only")
display(pd.concat([basic, major], axis="columns", join="inner"))

Outer alignment: all index labels


,Name,Major
101,Kevin,Mathematics
102,Tyler,NaN
103,Kyla,Computer Science
104,NaN,Biology


Inner alignment: shared index labels only


,Name,Major
101,Kevin,Mathematics
103,Kyla,Computer Science


<div style="background:#fffaf0;border-left:4px solid #D9B45B;padding:11px 16px;border-radius:4px;margin:10px 0"><b style="color:#614616">Critical distinction</b><br>The index is <b>not</b> a student identifier unless you intentionally make it one. Horizontal <code>concat()</code> does not look for matching <code>Name</code> or <code>SID</code> column values. Use <code>merge()</code> for value-based matching.</div>

<div style="border-left:6px solid #512888;padding:13px 18px;background:#f4f0fa;border-radius:5px;margin:24px 0 14px 0"><div style="color:#512888;font-size:22px;font-weight:750">04 · Prepare two related tables</div><div style="color:#4b4655;font-size:14px;margin-top:5px">Use a common key to connect enrollment and student data</div></div>

In [8]:
enrollment = pd.DataFrame({
    "SID": [101, 102, 103, 104],
    "Name": ["Kelly", "John", "Megan", "Sara"],
    "Course": ["CSCI 1462", "CSCI 1462", "MATH 1304", "CSCI 1462"]
})
advisors = pd.DataFrame({
    "SID": [101, 102, 105],
    "Advisor": ["Dr. Lee", "Dr. Smith", "Dr. Patel"]
})
print("ENROLLMENT (left)"); display(enrollment)
print("ADVISORS (right)"); display(advisors)

ENROLLMENT (left)


,SID,Name,Course
0,101,Kelly,CSCI 1462
1,102,John,CSCI 1462
2,103,Megan,MATH 1304
3,104,Sara,CSCI 1462


ADVISORS (right)


,SID,Advisor
0,101,Dr. Lee
1,102,Dr. Smith
2,105,Dr. Patel


The `SID` column is our **join key**. `101` and `102` appear in both tables; `103` and `104` appear only on the left; `105` appears only on the right. The examples below use the *same inputs*, so you can compare their outcomes directly.

<div style="background:#fff;border:1px solid #e1d9ea;border-radius:10px;padding:17px 19px;margin:15px 0;">
 <div style="font-size:18px;color:#512888;font-weight:750;margin-bottom:8px;">Join types at a glance</div>
 <div style="color:#4f4958;line-height:1.5;margin-bottom:10px;">For the sample tables below: <b>left keys = &#123;101, 102, 103, 104&#125;</b> and <b>right keys = &#123;101, 102, 105&#125;</b>. The examples have unique IDs, so each matching key produces one row.</div>
 <table style="border-collapse:collapse;width:100%;font-size:14px;">
  <thead><tr style="background:#512888;color:white;"><th style="padding:10px;text-align:left;">Join</th><th style="padding:10px;text-align:left;">Keys retained</th><th style="padding:10px;text-align:left;">Intuition</th></tr></thead>
  <tbody>
   <tr style="background:#f6f1fb;"><td style="padding:9px;border-bottom:1px solid #eee;">INNER</td><td style="padding:9px;border-bottom:1px solid #eee;"><b>101, 102</b></td><td style="padding:9px;border-bottom:1px solid #eee;">Intersection &cap;</td></tr>
   <tr><td style="padding:9px;border-bottom:1px solid #eee;">LEFT</td><td style="padding:9px;border-bottom:1px solid #eee;"><b>101, 102, 103, 104</b></td><td style="padding:9px;border-bottom:1px solid #eee;">All left keys</td></tr>
   <tr style="background:#f6f1fb;"><td style="padding:9px;border-bottom:1px solid #eee;">RIGHT</td><td style="padding:9px;border-bottom:1px solid #eee;"><b>101, 102, 105</b></td><td style="padding:9px;border-bottom:1px solid #eee;">All right keys</td></tr>
   <tr><td style="padding:9px;">OUTER</td><td style="padding:9px;"><b>101, 102, 103, 104, 105</b></td><td style="padding:9px;">Union &cup;</td></tr>
  </tbody>
 </table>
</div>

<div style="border-left:6px solid #512888;padding:13px 18px;background:#f4f0fa;border-radius:5px;margin:24px 0 14px 0"><div style="color:#512888;font-size:22px;font-weight:750">05 · Inner join</div><div style="color:#4b4655;font-size:14px;margin-top:5px">Keep only records with matching keys in both tables</div></div>

In [9]:
inner_result = pd.merge(enrollment, advisors, how="inner", on="SID")
display(inner_result)

,SID,Name,Course,Advisor
0,101,Kelly,CSCI 1462,Dr. Lee
1,102,John,CSCI 1462,Dr. Smith


An **inner join** keeps the intersection of key values: here, only `SID` 101 and 102. `pd.merge()` defaults to `how="inner"`, but specifying it makes the intent clearer.

<div style="border-left:6px solid #512888;padding:13px 18px;background:#f4f0fa;border-radius:5px;margin:24px 0 14px 0"><div style="color:#512888;font-size:22px;font-weight:750">06 · Left join</div><div style="color:#4b4655;font-size:14px;margin-top:5px">Keep all left-side records</div></div>

In [10]:
left_result = pd.merge(enrollment, advisors, how="left", on="SID")
display(left_result)

,SID,Name,Course,Advisor
0,101,Kelly,CSCI 1462,Dr. Lee
1,102,John,CSCI 1462,Dr. Smith
2,103,Megan,MATH 1304,NaN
3,104,Sara,CSCI 1462,NaN


Every enrollment row is kept. Students without a matching advisor show a missing value in `Advisor`. A left join is useful when the first table defines the population you want to study.

<div style="border-left:6px solid #512888;padding:13px 18px;background:#f4f0fa;border-radius:5px;margin:24px 0 14px 0"><div style="color:#512888;font-size:22px;font-weight:750">07 · Right join</div><div style="color:#4b4655;font-size:14px;margin-top:5px">Keep all right-side records</div></div>

In [11]:
right_result = pd.merge(enrollment, advisors, how="right", on="SID")
display(right_result)

,SID,Name,Course,Advisor
0,101,Kelly,CSCI 1462,Dr. Lee
1,102,John,CSCI 1462,Dr. Smith
2,105,NaN,NaN,Dr. Patel


Every advisor row is kept, including `SID` 105, who is not in the enrollment table. Note that the `Name` and `Course` columns are missing for that record.

<div style="border-left:6px solid #512888;padding:13px 18px;background:#f4f0fa;border-radius:5px;margin:24px 0 14px 0"><div style="color:#512888;font-size:22px;font-weight:750">08 · Outer join</div><div style="color:#4b4655;font-size:14px;margin-top:5px">Keep all keys from either table</div></div>

In [12]:
outer_result = pd.merge(enrollment, advisors, how="outer", on="SID", indicator=True)
display(outer_result)

,SID,Name,Course,Advisor,_merge
0,101,Kelly,CSCI 1462,Dr. Lee,both
1,102,John,CSCI 1462,Dr. Smith,both
2,103,Megan,MATH 1304,NaN,left_only
3,104,Sara,CSCI 1462,NaN,left_only
4,105,NaN,NaN,Dr. Patel,right_only


An **outer join** retains the union of key values. `indicator=True` adds `_merge`, which reveals whether a record came from the left only, right only, or both. This is especially useful when checking data quality.

In [13]:
print("Merge source counts")
display(outer_result["_merge"].value_counts())

Merge source counts


_merge
left_only     2
both          2
right_only    1
Name: count, dtype: int64

<div style="background:#fff8e9;border-left:5px solid #e3b747;border-radius:7px;padding:13px 18px;color:#3e3522;line-height:1.62;">
 <b style="color:#674b11;">Union and intersection refer to keys, not automatic row deduplication.</b><br>
 When a join key occurs more than once, pandas returns every matching row combination. Consequently, an inner join can contain more rows than either source. The duplicate-key demonstration later in this notebook makes this behavior concrete.
</div>

<div style="border-left:6px solid #512888;padding:13px 18px;background:#f4f0fa;border-radius:5px;margin:24px 0 14px 0"><div style="color:#512888;font-size:22px;font-weight:750">09 · Cross join (Cartesian product)</div><div style="color:#4b4655;font-size:14px;margin-top:5px">Pair every left row with every right row</div></div>

In [14]:
courses = pd.DataFrame({"Course": ["CSCI 1462", "CSCI 3302"]})
rooms = pd.DataFrame({"Room": ["STEM 417", "STEM 420", "STEM 425"]})
print("COURSES"); display(courses)
print("ROOMS"); display(rooms)
combinations = pd.merge(courses, rooms, how="cross")
print("CROSS JOIN"); display(combinations)
print("Number of rows:", len(combinations), "=", len(courses), "×", len(rooms))

COURSES


,Course
0,CSCI 1462
1,CSCI 3302


ROOMS


,Room
0,STEM 417
1,STEM 420
2,STEM 425


CROSS JOIN


,Course,Room
0,CSCI 1462,STEM 417
1,CSCI 1462,STEM 420
2,CSCI 1462,STEM 425
3,CSCI 3302,STEM 417
4,CSCI 3302,STEM 420
5,CSCI 3302,STEM 425


Number of rows: 6 = 2 × 3


<div style="background:#fffaf0;border-left:4px solid #D9B45B;padding:11px 16px;border-radius:4px;margin:10px 0"><b style="color:#614616">Be careful</b><br>A cross join between tables containing <i>m</i> and <i>n</i> rows produces <i>m × n</i> rows. A cross join has no <code>on=</code> key; large tables can produce enormous results.</div>

<div style="border-left:6px solid #512888;padding:13px 18px;background:#f4f0fa;border-radius:5px;margin:24px 0 14px 0"><div style="color:#512888;font-size:22px;font-weight:750">10 · Common pitfall: duplicate keys</div><div style="color:#4b4655;font-size:14px;margin-top:5px">A merge can multiply rows</div></div>

In [15]:
registrations = pd.DataFrame({
    "SID": [101, 101, 102],
    "Course": ["CSCI 1462", "CSCI 3302", "MATH 1304"]
})
contacts = pd.DataFrame({
    "SID": [101, 101, 102],
    "Email": ["kelly@school.edu", "kelly.alt@school.edu", "john@school.edu"]
})
print("REGISTRATIONS"); display(registrations)
print("CONTACTS"); display(contacts)
print("MERGED RESULT"); display(pd.merge(registrations, contacts, on="SID"))

REGISTRATIONS


,SID,Course
0,101,CSCI 1462
1,101,CSCI 3302
2,102,MATH 1304


CONTACTS


,SID,Email
0,101,kelly@school.edu
1,101,kelly.alt@school.edu
2,102,john@school.edu


MERGED RESULT


,SID,Course,Email
0,101,CSCI 1462,kelly@school.edu
1,101,CSCI 1462,kelly.alt@school.edu
2,101,CSCI 3302,kelly@school.edu
3,101,CSCI 3302,kelly.alt@school.edu
4,102,MATH 1304,john@school.edu


`SID=101` appears twice in each table, so the merge produces **2 × 2 = 4 rows** for that student. This is a **many-to-many join**, not a pandas bug. Before joining, check whether the key is unique in either table.

<div style="border-left:6px solid #512888;padding:13px 18px;background:#f4f0fa;border-radius:5px;margin:24px 0 14px 0"><div style="color:#512888;font-size:22px;font-weight:750">11 · Merge on more than one column</div><div style="color:#4b4655;font-size:14px;margin-top:5px">Prevent incorrect matches when one field is not enough</div></div>

In [18]:
people = pd.DataFrame({"FirstName": ["Mary", "Vincent", "Jack"],
                       "LastName": ["Dufrene", "Robin", "Sparrow"]})
cities = pd.DataFrame({"FirstName": ["Jack", "Mary", "Vincent", "Jack"],
                       "LastName": ["Sullivan", "Dufrene", "Morgan", "Sparrow"],
                       "City": ["Nacogdoches", "Houston", "Dallas", "Houston"]})
print("PEOPLE"); display(people)
print("CITIES"); display(cities)
print("Match on first name ONLY (can match wrong people):")
display(pd.merge(people, cities, how="inner", on="FirstName"))
print("Match on first AND last name:")
display(pd.merge(people, cities, how="inner", on=["FirstName", "LastName"]))

PEOPLE


,FirstName,LastName
0,Mary,Dufrene
1,Vincent,Robin
2,Jack,Sparrow


CITIES


,FirstName,LastName,City
0,Jack,Sullivan,Nacogdoches
1,Mary,Dufrene,Houston
2,Vincent,Morgan,Dallas
3,Jack,Sparrow,Houston


Match on first name ONLY (can match wrong people):


,FirstName,LastName_x,LastName_y,City
0,Mary,Dufrene,Dufrene,Houston
1,Vincent,Robin,Morgan,Dallas
2,Jack,Sparrow,Sullivan,Nacogdoches
3,Jack,Sparrow,Sparrow,Houston


Match on first AND last name:


,FirstName,LastName,City
0,Mary,Dufrene,Houston
1,Jack,Sparrow,Houston


When a single field is ambiguous, supply a list of keys: `on=["FirstName", "LastName"]`. In real datasets, a stable unique identifier such as a student ID is preferable to names.

<div style="border-left:6px solid #512888;padding:13px 18px;background:#f4f0fa;border-radius:5px;margin:24px 0 14px 0"><div style="color:#512888;font-size:22px;font-weight:750">12 · Different key names and overlapping columns</div><div style="color:#4b4655;font-size:14px;margin-top:5px">Use left_on, right_on, and suffixes</div></div>

In [19]:
grades = pd.DataFrame({"StudentID": [101, 102, 103], "Status": ["Passed", "Passed", "In progress"]})
housing = pd.DataFrame({"SID": [101, 103, 104], "Status": ["On-campus", "Off-campus", "On-campus"]})
display(pd.merge(grades, housing, how="left", left_on="StudentID", right_on="SID"))

,StudentID,Status_x,SID,Status_y
0,101,Passed,101.0,On-campus
1,102,Passed,NaN,NaN
2,103,In progress,103.0,Off-campus


`left_on` and `right_on` tell pandas which differently named columns contain corresponding keys. `suffixes` distinguishes non-key column names that occur in both tables.

<div style="border-left:6px solid #512888;padding:13px 18px;background:#f4f0fa;border-radius:5px;margin:24px 0 14px 0"><div style="color:#512888;font-size:22px;font-weight:750">13 · Missing keys and merge diagnostics</div><div style="color:#4b4655;font-size:14px;margin-top:5px">Understand NaN and check output size</div></div>

Missing values in **non-key** columns often arise naturally from unmatched rows in left, right, or outer joins. However, pandas also treats **null keys as matching one another** during a merge. Inspect missing join keys before merging.

In [20]:
a = pd.DataFrame({"SID": [101, None], "Source": ["A", "Unknown A"]})
b = pd.DataFrame({"SID": [101, None], "Group": ["Blue", "Unassigned"]})
print("Notice that the null-key rows match:")
display(pd.merge(a, b, on="SID", how="inner"))

Notice that the null-key rows match:


,SID,Source,Group
0,101.0,A,Blue
1,NaN,Unknown A,Unassigned


<div style="background:#fffaf0;border-left:4px solid #D9B45B;padding:11px 16px;border-radius:4px;margin:10px 0"><b style="color:#614616">Good practice</b><br>For every important merge, inspect key uniqueness, check missing key values, choose the join type deliberately, compare row counts, and use <code>indicator=True</code> or <code>validate=...</code> when appropriate.</div>

<div style="border-left:6px solid #512888;padding:13px 18px;background:#f4f0fa;border-radius:5px;margin:24px 0 14px 0"><div style="color:#512888;font-size:22px;font-weight:750">15 · Quick reference</div><div style="color:#4b4655;font-size:14px;margin-top:5px">Choose the right pandas operation</div></div>

| Your goal | Recommended pandas operation |
|---|---|
| Stack same-structure tables vertically | `pd.concat([a, b], ignore_index=True)` |
| Align columns on DataFrame index | `pd.concat([a, b], axis=1)` |
| Keep only matching keys | `pd.merge(a, b, on="ID", how="inner")` |
| Keep all left-side records | `pd.merge(a, b, on="ID", how="left")` |
| Keep all right-side records | `pd.merge(a, b, on="ID", how="right")` |
| Keep all keys from both tables | `pd.merge(a, b, on="ID", how="outer")` |
| Pair every possible combination | `pd.merge(a, b, how="cross")` |
| Match on multiple fields | `pd.merge(a, b, on=["First", "Last"])` |

**Key takeaway:** `concat` aligns on labels; `merge` matches on key values. Always verify how many rows your result contains.

<div style="background:#512888;color:white;border-left:8px solid #e3b747;border-radius:9px;padding:20px 23px;margin-top:19px;">
 <div style="font-size:20px;font-weight:780;color:white;">Takeaway</div>
 <div style="margin-top:8px;line-height:1.65;color:#f5eaff;">Use <b>concat()</b> to combine table blocks. Use <b>merge()</b> to relate records by keys. Before trusting a result, inspect the matching columns, row counts, duplicate keys, and missing values.</div>
</div>